# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [8]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

### Total Number of Flows

Count the total number of flows in this trace.

In [9]:
flows = ndf.groupby(['Source', 'Destination', 'Protocol'])
print(f"Total number of flows: {flows.ngroups}")

Total number of flows: 148


### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [4]:
flow_bytes = flows['Length'].sum().sort_values(ascending=False)
flow_bytes.head(10)

Source                                              Destination                                     
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                       120607242
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                         7138148
192.168.43.72                                       ipv4-c071-cdg001-ix.1.oca.nflxvideo.net               3357228
a23-57-80-120.deploy.static.akamaitechnologies.com  192.168.43.72                                         1332086
ipv4-c063-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                          431178
ec2-52-19-39-146.eu-west-1.compute.amazonaws.com    192.168.43.72                                          348141
192.168.43.72                                       ec2-52-19-39-146.eu-west-1.compute.amazonaws.com       340330
                                                    ipv4-c069-cdg001-ix.1.oca.nflxvideo.net          

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [5]:
flow_packets = flows['Length'].count().sort_values(ascending=False)
flow_packets.head(10)

Source                                              Destination                                       
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                         80084
192.168.43.72                                       ipv4-c071-cdg001-ix.1.oca.nflxvideo.net               47902
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net             192.168.43.72                                          4873
192.168.43.72                                       ipv4-c069-cdg001-ix.1.oca.nflxvideo.net                3170
a23-57-80-120.deploy.static.akamaitechnologies.com  192.168.43.72                                          1005
192.168.43.72                                       a23-57-80-120.deploy.static.akamaitechnologies.com      834
                                                    ec2-52-19-39-146.eu-west-1.compute.amazonaws.com        489
ec2-52-19-39-146.eu-west-1.compute.amazonaws.com    192.168.43.72                                           472
i

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [6]:
ndf['Time'] = pd.to_datetime(ndf['Time'])

flow_duration = flows['Time'].apply(lambda x: x.max() - x.min())
flow_duration.sort_values(ascending=False).head(10)

Source                                            Destination                                     
192.168.43.72                                     par10s38-in-f3.1e100.net                           0 days 00:08:15.451688
par10s38-in-f3.1e100.net                          192.168.43.72                                      0 days 00:08:15.104425
ns-vip-pro.paris.inria.fr                         192.168.43.72                                      0 days 00:08:12.745875
192.168.43.72                                     ns-vip-pro.paris.inria.fr                          0 days 00:08:12.687393
192.168.43.97                                     224.0.0.251                                        0 days 00:08:06.705466
192.168.43.72                                     224.0.0.251                                        0 days 00:08:06.400795
fe80::e6ce:8fff:fe01:4c54                         ff02::fb                                           0 days 00:08:06.400733
192.168.43.72                    

## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

In [7]:
flow_stats = pd.DataFrame({
    'bytes': flows['Length'].sum(),
    'packets': flows['Length'].count(),
    'duration_sec': flows['Time'].apply(lambda x: (x.max() - x.min()).total_seconds())
})

flow_stats['bytes_per_sec'] = flow_stats['bytes'] / flow_stats['duration_sec'].replace(0, float('nan'))
flow_stats['packets_per_sec'] = flow_stats['packets'] / flow_stats['duration_sec'].replace(0, float('nan'))

flow_stats.sort_values('bytes_per_sec', ascending=False).head(10)

,,bytes,packets,duration_sec,bytes_per_sec,packets_per_sec
Source,Destination,,,,,
192.168.1.159,192.168.43.72,752,8,0.000827,909310.761790,9673.518742
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,120607242,80084,475.340378,253728.165294,168.477166
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,7138148,4873,475.341855,15016.872436,10.251569
192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,3357228,47902,475.478319,7060.738347,100.744867
ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,431178,338,75.093257,5741.900368,4.501070
a23-57-80-120.deploy.static.akamaitechnologies.com,192.168.43.72,1332086,1005,372.896515,3572.267228,2.695118
198.38.120.153,192.168.43.72,39507,63,13.751028,2873.021566,4.581476
par21s03-in-f2.1e100.net,192.168.43.72,6897,15,2.686329,2567.444271,5.583828
198.38.120.137,192.168.43.72,116255,101,67.323224,1726.818668,1.500225


## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?